<center><h1>Wagner_Anton_HW7</h1></center>

Name: Anton Wagner
<br>
Github Username: AntonW1
<br>
USC ID: 2245157283

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.model_selection import GridSearchCV, StratifiedKFold
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, roc_auc_score, RocCurveDisplay

In [19]:
!pip install imbalanced-learn

In [20]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
data_path = '../Homework 7 Data/Anuran Calls (MFCCs)/Frogs_MFCCs.csv'

df = pd.read_csv(data_path)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7195 entries, 0 to 7194
Data columns (total 26 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MFCCs_ 1  7195 non-null   float64
 1   MFCCs_ 2  7195 non-null   float64
 2   MFCCs_ 3  7195 non-null   float64
 3   MFCCs_ 4  7195 non-null   float64
 4   MFCCs_ 5  7195 non-null   float64
 5   MFCCs_ 6  7195 non-null   float64
 6   MFCCs_ 7  7195 non-null   float64
 7   MFCCs_ 8  7195 non-null   float64
 8   MFCCs_ 9  7195 non-null   float64
 9   MFCCs_10  7195 non-null   float64
 10  MFCCs_11  7195 non-null   float64
 11  MFCCs_12  7195 non-null   float64
 12  MFCCs_13  7195 non-null   float64
 13  MFCCs_14  7195 non-null   float64
 14  MFCCs_15  7195 non-null   float64
 15  MFCCs_16  7195 non-null   float64
 16  MFCCs_17  7195 non-null   float64
 17  MFCCs_18  7195 non-null   float64
 18  MFCCs_19  7195 non-null   float64
 19  MFCCs_20  7195 non-null   float64
 20  MFCCs_21  7195 non-null   float64
 21  MF

In [4]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['Family', 'Genus', 'Species', 'RecordID']), df[['Family', 'Genus', 'Species']], train_size=0.7, random_state=552)

### (b) Train a classifier for each label

#### (i) Research

The exact match approach for determining accuracy only counts predictions as correct if all target labels match the true labels. This means that partially correct predictions are not deferentiated from completely wrong predictions. Its loss is calculated by subtracting the exact match ratio from 1.

The hamming score approach counts the fraction of labels that the model predicted correctly. Hamming loss is the hamming score subtracted from 1.

#### (ii) Train a SVM for each of the labels

In [5]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [6]:
folds = StratifiedKFold(n_splits=10, random_state=552, shuffle=True)
C = [0.01, 0.05, 0.1, 0.5, 1, 10, 100, 500, 1000]
sigmas = [0.1, 0.25, 0.5, 0.75, 1, 2, 5, 10]
gammas = []
for sigma in sigmas:
    gamma = 1 / (2 * sigma ** 2)
    gammas.append(gamma)

preds = {}
for label in ['Family', 'Genus', 'Species']:
    model = SVC(kernel='rbf')
    SVM_model = OneVsRestClassifier(model)
    search_dict = {'estimator__C': C, 'estimator__gamma': gammas}
    grid_search = GridSearchCV(SVM_model, search_dict, scoring='accuracy', cv=folds, n_jobs=-2)
    grid_search.fit(x_train_scaled, y_train[label])
    preds[label] = grid_search.predict(x_test_scaled)

    accuracy = grid_search.best_score_
    C_best = grid_search.best_params_['estimator__C']
    gamma_best = grid_search.best_params_['estimator__gamma']
    print(label)
    print(f'Accuracy: {accuracy}')
    print(f'Best C: {C_best}')
    print(f'Best Gamma: {gamma_best}')
    print('')

Family
Accuracy: 0.9912615576382973
Best C: 100
Best Gamma: 0.02

Genus
Accuracy: 0.9896762283442204
Best C: 10
Best Gamma: 0.02

Species
Accuracy: 0.9880841932531794
Best C: 10
Best Gamma: 0.02



I picked a wide range of C and sigma values and the best values fell inside my test values.

In [8]:
preds

{'Family': array(['Leptodactylidae', 'Hylidae', 'Hylidae', ..., 'Bufonidae',
        'Leptodactylidae', 'Hylidae'], shape=(2159,), dtype='<U15'),
 'Genus': array(['Leptodactylus', 'Hypsiboas', 'Osteocephalus', ..., 'Rhinella',
        'Adenomera', 'Osteocephalus'], shape=(2159,), dtype='<U13'),
 'Species': array(['LeptodactylusFuscus', 'HypsiboasCordobae',
        'OsteocephalusOophagus', ..., 'Rhinellagranulosa',
        'AdenomeraHylaedactylus', 'OsteocephalusOophagus'],
       shape=(2159,), dtype='<U22')}

In [7]:
preds_df = pd.DataFrame(preds)
preds_df

,Family,Genus,Species
0,Leptodactylidae,Leptodactylus,LeptodactylusFuscus
1,Hylidae,Hypsiboas,HypsiboasCordobae
2,Hylidae,Osteocephalus,OsteocephalusOophagus
3,Hylidae,Hypsiboas,HypsiboasCordobae
4,Dendrobatidae,Ameerega,Ameeregatrivittata
...,...,...,...
2154,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
2155,Hylidae,Hypsiboas,HypsiboasCordobae
2156,Bufonidae,Rhinella,Rhinellagranulosa
2157,Leptodactylidae,Adenomera,AdenomeraHylaedactylus


In [9]:
incorrect_labels = np.array(preds_df) != np.array(y_test)
exact_match = np.mean(incorrect_labels.sum(axis=1) == 0)
hamming_loss = incorrect_labels.mean()
hamming_score = 1 - hamming_loss

print(f'Exact match: {exact_match}')
print(f'Hamming Loss: {hamming_loss}')
print(f'Hamming Score: {hamming_score}')

Exact match: 0.98471514590088
Hamming Loss: 0.011733827389223406
Hamming Score: 0.9882661726107766


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [12]:
folds = StratifiedKFold(n_splits=10, random_state=552, shuffle=True)
C = [0.01, 0.05, 0.1, 0.5, 1, 10, 100, 500, 1000]

print('SVM with L1 Penalty')
L1_preds = {}
for label in ['Family', 'Genus', 'Species']:
    L1_model = LinearSVC(penalty='l1', max_iter=5000, dual=False)
    search_dict = {'C': C}
    grid_search = GridSearchCV(L1_model, search_dict, scoring='accuracy', cv=folds, n_jobs=-2)
    grid_search.fit(x_train_scaled, y_train[label])
    L1_preds[label] = grid_search.predict(x_test_scaled)

    accuracy = grid_search.best_score_
    C_best = grid_search.best_params_['C']
    coefficients = grid_search.best_estimator_.coef_
    print(label)
    print(f'Accuracy: {accuracy}')
    print(f'Best C: {C_best}')
    print(f'Coefficients:\n{coefficients}')
    print('')

SVM with L1 Penalty
Family
Accuracy: 0.9378494903594307
Best C: 1
Coefficients:
[[ 0.01626929  0.05306772 -0.24095412 -0.28518093 -0.33634657 -0.35905195
  -0.03332119 -0.09422373 -0.12442239 -0.07556059  0.38036669 -0.09614151
   0.41224129 -0.17765836  0.10526671 -0.01045111  0.13590219 -0.1665467
   0.30039278  0.05193083 -0.07965019 -0.04167881]
 [ 0.30202368  0.18757379 -0.6760993  -0.33093955 -0.01415711 -0.17670675
  -0.13118531 -0.27435414  0.02693102 -0.53465254  0.38656373 -0.16915109
  -0.3986974   0.19833988  0.00225957  0.         -0.57389675  0.048641
   0.60980286 -0.73870213 -0.07745265 -0.03746307]
 [-0.11772232 -0.23992536  0.90142344 -0.45989563  0.59533386 -0.13488548
   0.01808895  0.23113271 -0.15220499  0.48671233 -0.21934938  0.52544959
  -0.47770995  0.01996377 -0.48794468  0.6439265  -0.57579782  0.65428627
  -0.27576876  0.71419965 -0.60291691  0.02799439]
 [ 0.05734097  0.06795393 -0.40787584  0.32936654 -0.43824331 -0.0966049
   0.07106463 -0.22287509  0.27

In [15]:
L1_preds

{'Family': array(['Leptodactylidae', 'Hylidae', 'Hylidae', ..., 'Hylidae',
        'Leptodactylidae', 'Hylidae'], shape=(2159,), dtype=object),
 'Genus': array(['Leptodactylus', 'Hypsiboas', 'Hypsiboas', ..., 'Rhinella',
        'Adenomera', 'Hypsiboas'], shape=(2159,), dtype=object),
 'Species': array(['LeptodactylusFuscus', 'HypsiboasCordobae',
        'OsteocephalusOophagus', ..., 'Rhinellagranulosa',
        'AdenomeraHylaedactylus', 'OsteocephalusOophagus'],
       shape=(2159,), dtype=object)}

In [16]:
L1_preds_df = pd.DataFrame(L1_preds)
L1_preds_df

,Family,Genus,Species
0,Leptodactylidae,Leptodactylus,LeptodactylusFuscus
1,Hylidae,Hypsiboas,HypsiboasCordobae
2,Hylidae,Hypsiboas,OsteocephalusOophagus
3,Hylidae,Hypsiboas,HypsiboasCordobae
4,Hylidae,Hypsiboas,HypsiboasCordobae
...,...,...,...
2154,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
2155,Hylidae,Hypsiboas,HypsiboasCordobae
2156,Hylidae,Rhinella,Rhinellagranulosa
2157,Leptodactylidae,Adenomera,AdenomeraHylaedactylus


In [18]:
L1_incorrect_labels = np.array(L1_preds_df) != np.array(y_test)
L1_exact_match = np.mean(L1_incorrect_labels.sum(axis=1) == 0)
L1_hamming_loss = L1_incorrect_labels.mean()
L1_hamming_score = 1 - L1_hamming_loss

print('SVM with L1 penalty')
print(f'Exact match: {L1_exact_match}')
print(f'Hamming Loss: {L1_hamming_loss}')
print(f'Hamming Score: {L1_hamming_score}')

Exact match: 0.9175544233441408
Hamming Loss: 0.05264783078585765
Hamming Score: 0.9473521692141423


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [22]:
print('SVM with SMOTE and L1 Penalty')
SMOTE_preds = {}
for label in ['Family', 'Genus', 'Species']:
    SMOTE_model = Pipeline([('SMOTE', SMOTE(random_state=552)), ('SVM', LinearSVC(penalty='l1', max_iter=5000, dual=False))])
    search_dict = {'SVM__C': C}
    grid_search = GridSearchCV(SMOTE_model, search_dict, scoring='accuracy', cv=folds, n_jobs=-2)
    grid_search.fit(x_train_scaled, y_train[label])
    SMOTE_preds[label] = grid_search.predict(x_test_scaled)

    accuracy = grid_search.best_score_
    C_best = grid_search.best_params_['SVM__C']
    print(label)
    print(f'Accuracy: {accuracy}')
    print(f'Best C: {C_best}')
    print('')

SVM with SMOTE and L1 Penalty
Family
Accuracy: 0.9225590899050143
Best C: 10

Genus
Accuracy: 0.9169984063870744
Best C: 1

Species
Accuracy: 0.9557188614345671
Best C: 0.1



In [24]:
SMOTE_preds_df = pd.DataFrame(SMOTE_preds)
SMOTE_preds_df

,Family,Genus,Species
0,Leptodactylidae,Leptodactylus,LeptodactylusFuscus
1,Hylidae,Hypsiboas,HypsiboasCordobae
2,Bufonidae,Osteocephalus,OsteocephalusOophagus
3,Hylidae,Hypsiboas,HypsiboasCordobae
4,Dendrobatidae,Ameerega,HypsiboasCordobae
...,...,...,...
2154,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
2155,Hylidae,Hypsiboas,HypsiboasCordobae
2156,Bufonidae,Rhinella,Rhinellagranulosa
2157,Leptodactylidae,Adenomera,AdenomeraHylaedactylus


In [25]:
SMOTE_incorrect_labels = np.array(SMOTE_preds_df) != np.array(y_test)
SMOTE_exact_match = np.mean(SMOTE_incorrect_labels.sum(axis=1) == 0)
SMOTE_hamming_loss = SMOTE_incorrect_labels.mean()
SMOTE_hamming_score = 1 - SMOTE_hamming_loss

print('SVM with SMOTE and with L1 penalty')
print(f'Exact match: {SMOTE_exact_match}')
print(f'Hamming Loss: {SMOTE_hamming_loss}')
print(f'Hamming Score: {SMOTE_hamming_score}')

SVM with SMOTE and with L1 penalty
Exact match: 0.8624363131079203
Hamming Loss: 0.07503473830477073
Hamming Score: 0.9249652616952293


The SVM using one vs all Gausian kernels performed the best out of the three methods. So, it seems like the curved decision boundary fits the data the best. The worst performing classifier was using SMOTE, which implies that class imbalance is not a major issue.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [33]:
x = df.drop(columns=['Family', 'Genus', 'Species', 'RecordID'])
y = df[['Family', 'Genus', 'Species']]

best_k_list = []
all_majority_labels = []
hamming_distances = []
hamming_losses = []
hamming_scores = []

for num in range(50):
    top_score = 0
    top_k = 0
    best_clusters = []
    for k in range(2, 51):
        k_means = KMeans(n_clusters=k, random_state=num, n_init=1)
        clusters = k_means.fit_predict(x)
        silhouette = silhouette_score(x, clusters, random_state=num, sample_size=1000)
        if silhouette > top_score:
            top_score = silhouette
            top_k = k
            best_clusters = clusters

    best_k_list.append(top_k)
    
    y_copy = y.copy()
    y_copy['cluster'] = best_clusters
    majority_label = y_copy.groupby('cluster').agg(lambda column: column.mode()[0])
    all_majority_labels.append(majority_label)
    predictions = majority_label.loc[best_clusters].values

    incorrect_labels = y.values != predictions
    hamming_distance = incorrect_labels.sum(axis=1).mean()
    hamming_loss = incorrect_labels.mean()
    hamming_score = 1 - hamming_loss
    hamming_distances.append(hamming_distance)
    hamming_losses.append(hamming_loss)
    hamming_scores.append(hamming_score)

### (b) Determine which family is the majority

In [45]:
print('The best K of each of the 50 runs:')
print(best_k_list)
print('\nAverage best K value:')
print(np.mean(best_k_list))

The best K of each of the 50 runs:
[4, 4, 4, 6, 3, 4, 4, 3, 3, 4, 4, 3, 4, 4, 3, 2, 4, 4, 2, 4, 6, 4, 4, 4, 6, 3, 4, 3, 3, 4, 2, 4, 3, 4, 6, 3, 6, 6, 2, 2, 2, 4, 6, 5, 4, 4, 3, 3, 3, 3]

Average best K value:
3.78


In [34]:
all_majority_labels

[                  Family      Genus                 Species
 cluster                                                    
 0        Leptodactylidae  Adenomera  AdenomeraHylaedactylus
 1                Hylidae  Hypsiboas       HypsiboasCordobae
 2                Hylidae  Hypsiboas    HypsiboasCinerascens
 3        Leptodactylidae  Adenomera          AdenomeraAndre,
                   Family      Genus                 Species
 cluster                                                    
 0        Leptodactylidae  Adenomera  AdenomeraHylaedactylus
 1                Hylidae  Hypsiboas       HypsiboasCordobae
 2        Leptodactylidae  Adenomera          AdenomeraAndre
 3                Hylidae  Hypsiboas       HypsiboasCordobae,
                   Family      Genus                 Species
 cluster                                                    
 0        Leptodactylidae  Adenomera  AdenomeraHylaedactylus
 1                Hylidae  Hypsiboas    HypsiboasCinerascens
 2        Leptodactyli

In [35]:
majority_table = pd.concat(all_majority_labels)
majority_table.value_counts()

Family           Genus          Species               
Hylidae          Hypsiboas      HypsiboasCordobae         61
Leptodactylidae  Adenomera      AdenomeraHylaedactylus    50
                                AdenomeraAndre            31
Hylidae          Hypsiboas      HypsiboasCinerascens      23
Dendrobatidae    Ameerega       Ameeregatrivittata        19
Hylidae          Leptodactylus  LeptodactylusFuscus        4
Leptodactylidae  Leptodactylus  LeptodactylusFuscus        1
Name: count, dtype: int64

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [37]:
print('KMeans Results')
print('')
print(f'Average Hamming distance: {np.mean(hamming_distances)}')
print(f'Standard deviation Hamming distance: {np.std(hamming_distances)}')
print('')
print(f'Average Hamming score: {np.mean(hamming_scores)}')
print(f'Standard deviation Hamming score: {np.std(hamming_scores)}')
print('')
print(f'Average Hamming loss: {np.mean(hamming_losses)}')
print(f'Standard deviation Hamming loss: {np.std(hamming_losses)}')

KMeans Results

Average Hamming distance: 0.7223127171646978
Standard deviation Hamming distance: 0.1275643403965338

Average Hamming score: 0.7592290942784342
Standard deviation Hamming score: 0.042521446798844606

Average Hamming loss: 0.24077090572156593
Standard deviation Hamming loss: 0.042521446798844606


## 3. ISLR 12.6.2

(a) See /charts/part_a_dendrogram_sketch for the sketch

(b) See /charts/part_b_dendrogram_sketch for the sketch

(c) The clusters would be observations 1 & 2 and observations 3 & 4 since they are separated by branches.

(d) The clusters would be observations 1, 2, 3 and observation 4.

(e) See /charts/part_e_dendrogram_sketch for the sketch